## The top 15 coins by market cap (excluding stablecoins) at each previous quarter-end from 2010

### 1: Get All Coins List and Stablecoin IDs

In [9]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import joblib

# Load API key
load_dotenv()
api_key = os.getenv("COINGECKO_API_KEY")

headers = {
    "accept": "application/json",
    "x-cg-pro-api-key": api_key
}

# Get all coins list
coins_list = requests.get("https://pro-api.coingecko.com/api/v3/coins/list", headers=headers).json()
joblib.dump(coins_list, "/Users/olaoluwatunmise/market-agent/database/all_coins_list.joblib")
print("Saved all_coins_list.joblib")

# Get stablecoin IDs
stablecoin_url = "https://pro-api.coingecko.com/api/v3/coins/markets"
stablecoin_params = {
    "vs_currency": "usd",
    "category": "stablecoins",
    "order": "market_cap_desc",
    "per_page": 250,
    "page": 1,
    "sparkline": "false"
}
stablecoin_response = requests.get(stablecoin_url, headers=headers, params=stablecoin_params)
stablecoin_data = stablecoin_response.json()
stablecoin_ids = set(coin["id"] for coin in stablecoin_data)
joblib.dump(stablecoin_ids, "/Users/olaoluwatunmise/market-agent/database/stablecoin_ids.joblib")
print("Saved /Users/olaoluwatunmise/market-agent/database/stablecoin_ids.joblib")

Saved all_coins_list.joblib
Saved /Users/olaoluwatunmise/market-agent/database/stablecoin_ids.joblib


### 2: Generate Quarter-End Dates

In [11]:
import pandas as pd
from datetime import timezone
import joblib

quarter_ends = pd.date_range(start='2020-03-31', end=pd.Timestamp.today(), freq='Q')
quarter_ends = [d.replace(tzinfo=timezone.utc) for d in quarter_ends]
joblib.dump(quarter_ends, "/Users/olaoluwatunmise/market-agent/database/quarter_ends.joblib")
print("Saved quarter_ends.joblib")

Saved quarter_ends.joblib


/var/folders/9f/dspzp56j3pjcftph22hs9gw40000gn/T/ipykernel_37136/615805672.py:5: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarter_ends = pd.date_range(start='2020-03-31', end=pd.Timestamp.today(), freq='Q')


In [13]:
quarter_ends = joblib.load("/Users/olaoluwatunmise/market-agent/database/quarter_ends.joblib")
print(quarter_ends)

[Timestamp('2020-03-31 00:00:00+0000', tz='UTC'), Timestamp('2020-06-30 00:00:00+0000', tz='UTC'), Timestamp('2020-09-30 00:00:00+0000', tz='UTC'), Timestamp('2020-12-31 00:00:00+0000', tz='UTC'), Timestamp('2021-03-31 00:00:00+0000', tz='UTC'), Timestamp('2021-06-30 00:00:00+0000', tz='UTC'), Timestamp('2021-09-30 00:00:00+0000', tz='UTC'), Timestamp('2021-12-31 00:00:00+0000', tz='UTC'), Timestamp('2022-03-31 00:00:00+0000', tz='UTC'), Timestamp('2022-06-30 00:00:00+0000', tz='UTC'), Timestamp('2022-09-30 00:00:00+0000', tz='UTC'), Timestamp('2022-12-31 00:00:00+0000', tz='UTC'), Timestamp('2023-03-31 00:00:00+0000', tz='UTC'), Timestamp('2023-06-30 00:00:00+0000', tz='UTC'), Timestamp('2023-09-30 00:00:00+0000', tz='UTC'), Timestamp('2023-12-31 00:00:00+0000', tz='UTC'), Timestamp('2024-03-31 00:00:00+0000', tz='UTC'), Timestamp('2024-06-30 00:00:00+0000', tz='UTC'), Timestamp('2024-09-30 00:00:00+0000', tz='UTC'), Timestamp('2024-12-31 00:00:00+0000', tz='UTC'), Timestamp('2025-03-

### 3: Fetch Price, Volume, and Market Cap for a Subset 

In [15]:
import requests
import pandas as pd
import time
import joblib
from datetime import timedelta
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
api_key = os.getenv("COINGECKO_API_KEY")

headers = {
    "accept": "application/json",
    "x-cg-pro-api-key": api_key
}

# Load previously saved data
stablecoin_ids = joblib.load("/Users/olaoluwatunmise/market-agent/database/stablecoin_ids.joblib")
quarter_ends = joblib.load("/Users/olaoluwatunmise/market-agent/database/quarter_ends.joblib")

# For testing, limit to a few quarters
#quarter_ends = quarter_ends[:4]  # first 4 quarters

# Get current top 100 coins by market cap
url = "https://pro-api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 100,
    "page": 1,
    "sparkline": "false"
}
response = requests.get(url, headers=headers, params=params)
current_top_coins = response.json()

# Convert to the format your script expects
coins_list = [{'id': coin['id'], 'symbol': coin['symbol'], 'name': coin['name']} 
              for coin in current_top_coins]

# Remove stablecoins from the list
coins_list = [coin for coin in coins_list if coin['id'] not in stablecoin_ids]

results = []

for q_date in quarter_ends:
    print(f"Processing quarter ending {q_date.date()}...")
    coin_data_list = []
    for coin in coins_list:
        coin_id = coin['id']
        
        # Remove this line since we already filtered stablecoins above
        # if coin_id in stablecoin_ids:
        #     continue  # skip stablecoins

        from_timestamp = int((q_date - timedelta(days=1)).timestamp())
        to_timestamp = int((q_date + timedelta(days=1)).timestamp())

        url = f"https://pro-api.coingecko.com/api/v3/coins/{coin_id}/market_chart/range"
        params = {
            "vs_currency": "usd",
            "from": from_timestamp,
            "to": to_timestamp
        }
        try:
            response = requests.get(url, headers=headers, params=params)
            if response.status_code != 200:
                print(f"Failed for {coin_id}: {response.status_code}")
                continue
            data = response.json()
            
            if all(k in data and data[k] for k in ['market_caps', 'total_volumes', 'prices']):
                def closest(lst): return min(lst, key=lambda x: abs(x[0]/1000 - q_date.timestamp()))
                market_cap = closest(data['market_caps'])[1]
                volume = closest(data['total_volumes'])[1]
                price = closest(data['prices'])[1]
                
                coin_data_list.append({
                    'id': coin_id,
                    'symbol': coin['symbol'],
                    'name': coin['name'],
                    'market_cap': market_cap,
                    'volume': volume,
                    'price': price,
                    'quarter_end': q_date.date()
                })
        except Exception as e:
            print(f"Error for {coin_id}: {e}")
        time.sleep(1.1)

    # Sort and select top 15 by market cap for this quarter
    if coin_data_list:
        df = pd.DataFrame(coin_data_list)
        df = df.sort_values('market_cap', ascending=False).head(15)
        df['quarter_end'] = q_date.date()
        results.append(df)
        print(f"Top 15 for {q_date.date()}: {list(df['symbol'])}")
    else:
        print(f"No data found for quarter ending {q_date.date()}")

# Save results
if results:
    final_df = pd.concat(results, ignore_index=True)
    joblib.dump(final_df, "/Users/olaoluwatunmise/market-agent/database/top15_quarterly_data.joblib")
    print(f"Saved top15_quarterly_data.joblib with {len(final_df)} records")
    print(f"Data covers {len(results)} quarters")

Processing quarter ending 2020-03-31...
Top 15 for 2020-03-31: ['btc', 'eth', 'xrp', 'bch', 'ltc', 'bnb', 'okb', 'leo', 'ada', 'xmr', 'xlm', 'link', 'trx', 'cro', 'etc']
Processing quarter ending 2020-06-30...
Top 15 for 2020-06-30: ['btc', 'eth', 'xrp', 'bch', 'ltc', 'ada', 'bnb', 'cro', 'link', 'okb', 'xlm', 'leo', 'xmr', 'trx', 'etc']
Processing quarter ending 2020-09-30...
Top 15 for 2020-09-30: ['btc', 'eth', 'xrp', 'bnb', 'bch', 'dot', 'link', 'ada', 'cro', 'ltc', 'trx', 'xmr', 'okb', 'xlm', 'atom']
Processing quarter ending 2020-12-31...
Top 15 for 2020-12-31: ['btc', 'eth', 'xrp', 'ltc', 'dot', 'bch', 'ada', 'bnb', 'link', 'xlm', 'xmr', 'okb', 'trx', 'leo', 'cro']
Processing quarter ending 2021-03-31...
Top 15 for 2021-03-31: ['btc', 'eth', 'bnb', 'ada', 'dot', 'xrp', 'uni', 'ltc', 'link', 'bch', 'fil', 'xlm', 'doge', 'vet', 'cro']
Processing quarter ending 2021-06-30...
Top 15 for 2021-06-30: ['btc', 'eth', 'bnb', 'ada', 'doge', 'xrp', 'dot', 'bch', 'uni', 'ltc', 'sol', 'link'